In [2]:
import pandas as pd
import numpy as np
import ast
import nltk
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
import pickle

In [3]:
movies = pd.read_csv('movies_metadata.csv')

/var/folders/2q/w7nzh7ds7vs98_jcrgbw82v00000gn/T/ipykernel_5469/2992503378.py:1: DtypeWarning: Columns (0: popularity) have mixed types. Specify dtype option on import or set low_memory=False.
  movies = pd.read_csv('movies_metadata.csv')


In [4]:
movies.head(1)

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0


In [5]:
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  str    
 1   belongs_to_collection  4494 non-null   str    
 2   budget                 45466 non-null  str    
 3   genres                 45466 non-null  str    
 4   homepage               7782 non-null   str    
 5   id                     45466 non-null  str    
 6   imdb_id                45449 non-null  str    
 7   original_language      45455 non-null  str    
 8   original_title         45466 non-null  str    
 9   overview               44512 non-null  str    
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  str    
 12  production_companies   45463 non-null  str    
 13  production_countries   45463 non-null  str    
 14  release_date           45379 non-null  str    
 15  revenue      

In [6]:
df = movies[['id','title','adult','belongs_to_collection','genres','overview','poster_path','production_companies','tagline',]]

In [7]:
df.head(1)

,id,title,adult,belongs_to_collection,genres,overview,poster_path,production_companies,tagline
0,862,Toy Story,False,"{'id': 10194, 'name': 'Toy Story Collection', ...","[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","Led by Woody, Andy's toys live happily in his ...",/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,"[{'name': 'Pixar Animation Studios', 'id': 3}]",NaN


In [8]:
df.isnull().sum()


id                           0
title                        6
adult                        0
belongs_to_collection    40972
genres                       0
overview                   954
poster_path                386
production_companies         3
tagline                  25054
dtype: int64

In [9]:
df[['belongs_to_collection','overview','tagline']] = df[['belongs_to_collection','overview','tagline']].fillna(' ')

In [10]:
df = df.dropna()

In [11]:
df.head(1)

,id,title,adult,belongs_to_collection,genres,overview,poster_path,production_companies,tagline
0,862,Toy Story,False,"{'id': 10194, 'name': 'Toy Story Collection', ...","[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","Led by Woody, Andy's toys live happily in his ...",/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,"[{'name': 'Pixar Animation Studios', 'id': 3}]",


In [12]:
df['genres'][0]

"[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]"

In [13]:
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

In [14]:
def convertCol(obj):
    if not obj or obj == " ":
        return []
    
    try:
        # Convert string to list/dict
        data = ast.literal_eval(obj)
        L = []
        
        # If it's a list, iterate through each dictionary
        if isinstance(data, list):
            for i in data:
                if isinstance(i, dict) and 'name' in i:
                    L.append(i['name'])
        # If it's a single dictionary, just grab the name
        elif isinstance(data, dict) and 'name' in data:
            L.append(data['name'])
            
        return L
    except:
        return []

In [15]:
df.genres = df.genres.apply(convert)
df['production_companies'] = df['production_companies'].apply(convert)


In [17]:
df['belongs_to_collection'] = df['belongs_to_collection'].apply(convertCol)

In [18]:
df.head()

,id,title,adult,belongs_to_collection,genres,overview,poster_path,production_companies,tagline
0,862,Toy Story,False,[Toy Story Collection],"[Animation, Comedy, Family]","Led by Woody, Andy's toys live happily in his ...",/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,[Pixar Animation Studios],
1,8844,Jumanji,False,[],"[Adventure, Fantasy, Family]",When siblings Judy and Peter discover an encha...,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,"[TriStar Pictures, Teitler Film, Interscope Co...",Roll the dice and unleash the excitement!
2,15602,Grumpier Old Men,False,[Grumpy Old Men Collection],"[Romance, Comedy]",A family wedding reignites the ancient feud be...,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg,"[Warner Bros., Lancaster Gate]",Still Yelling. Still Fighting. Still Ready for...
3,31357,Waiting to Exhale,False,[],"[Comedy, Drama, Romance]","Cheated on, mistreated and stepped on, the wom...",/16XOMpEaLWkrcPqSQqhTmeJuqQl.jpg,[Twentieth Century Fox Film Corporation],Friends are the people who let you be yourself...
4,11862,Father of the Bride Part II,False,[Father of the Bride Collection],[Comedy],Just when George Banks has recovered from his ...,/e64sOI48hQXyru7naBFyssKFxVd.jpg,"[Sandollar Productions, Touchstone Pictures]",Just When His World Is Back To Normal... He's ...


In [19]:
df = df.drop(columns=['adult'])

In [20]:
df.overview = df.overview.apply(lambda x:x.split())
df.tagline = df.tagline.apply(lambda x:x.split())

In [21]:
df.head(1)

,id,title,belongs_to_collection,genres,overview,poster_path,production_companies,tagline
0,862,Toy Story,[Toy Story Collection],"[Animation, Comedy, Family]","[Led, by, Woody,, Andy's, toys, live, happily,...",/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,[Pixar Animation Studios],[]


In [22]:
df.genres = df.genres.apply(lambda x:[i.replace(" ","") for i in x])
df['belongs_to_collection'] = df['belongs_to_collection'].apply(lambda x:[i.replace(" ","") for i in x])
df['production_companies'] = df['production_companies'].apply(lambda x:[i.replace(" ","") for i in x])

In [23]:
df['tags'] =   df.genres+ df['belongs_to_collection']+  df['belongs_to_collection']+ df.production_companies +df.production_companies + df.overview +  df.tagline

In [24]:
df.head(1)

,id,title,belongs_to_collection,genres,overview,poster_path,production_companies,tagline,tags
0,862,Toy Story,[ToyStoryCollection],"[Animation, Comedy, Family]","[Led, by, Woody,, Andy's, toys, live, happily,...",/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,[PixarAnimationStudios],[],"[Animation, Comedy, Family, ToyStoryCollection..."


In [25]:
data = df[['id','title','poster_path','tags']]

In [26]:
data.head()

,id,title,poster_path,tags
0,862,Toy Story,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,"[Animation, Comedy, Family, ToyStoryCollection..."
1,8844,Jumanji,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,"[Adventure, Fantasy, Family, TriStarPictures, ..."
2,15602,Grumpier Old Men,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg,"[Romance, Comedy, GrumpyOldMenCollection, Grum..."
3,31357,Waiting to Exhale,/16XOMpEaLWkrcPqSQqhTmeJuqQl.jpg,"[Comedy, Drama, Romance, TwentiethCenturyFoxFi..."
4,11862,Father of the Bride Part II,/e64sOI48hQXyru7naBFyssKFxVd.jpg,"[Comedy, FatheroftheBrideCollection, Fatheroft..."


In [27]:
data.tags = data.tags.apply(lambda x:" ".join(x))
data.tags = data.tags.apply(lambda x:x.lower())

In [28]:
ps = PorterStemmer()

In [29]:
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

In [30]:
data.tags = data.tags.apply(stem)

In [31]:
data.head()

,id,title,poster_path,tags
0,862,Toy Story,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,anim comedi famili toystorycollect toystorycol...
1,8844,Jumanji,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,adventur fantasi famili tristarpictur teitlerf...
2,15602,Grumpier Old Men,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg,romanc comedi grumpyoldmencollect grumpyoldmen...
3,31357,Waiting to Exhale,/16XOMpEaLWkrcPqSQqhTmeJuqQl.jpg,comedi drama romanc twentiethcenturyfoxfilmcor...
4,11862,Father of the Bride Part II,/e64sOI48hQXyru7naBFyssKFxVd.jpg,comedi fatherofthebridecollect fatherofthebrid...


In [32]:
data.info()

<class 'pandas.DataFrame'>
Index: 45077 entries, 0 to 45465
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           45077 non-null  str  
 1   title        45077 non-null  str  
 2   poster_path  45077 non-null  str  
 3   tags         45077 non-null  str  
dtypes: str(4)
memory usage: 21.7 MB


In [33]:
data = data.drop_duplicates()

In [34]:
data = data.reset_index(drop = True)

In [35]:
indices = pd.Series(data.index,index= data['title']).drop_duplicates().to_dict()

In [36]:
indices

{'Toy Story': 0,
 'Jumanji': 1,
 'Grumpier Old Men': 2,
 'Waiting to Exhale': 3,
 'Father of the Bride Part II': 4,
 'Heat': 28866,
 'Sabrina': 883,
 'Tom and Huck': 7,
 'Sudden Death': 8,
 'GoldenEye': 9,
 'The American President': 10,
 'Dracula: Dead and Loving It': 11,
 'Balto': 12,
 'Nixon': 13,
 'Cutthroat Island': 14,
 'Casino': 15,
 'Sense and Sensibility': 40699,
 'Four Rooms': 17,
 'Ace Ventura: When Nature Calls': 18,
 'Money Train': 19,
 'Get Shorty': 20,
 'Copycat': 21,
 'Assassins': 22,
 'Powder': 23,
 'Leaving Las Vegas': 24,
 'Othello': 21155,
 'Now and Then': 26,
 'Persuasion': 40496,
 'The City of Lost Children': 28,
 'Shanghai Triad': 29,
 'Dangerous Minds': 30,
 'Twelve Monkeys': 31,
 'Wings of Courage': 32,
 'Babe': 33,
 'Carrington': 34,
 'Dead Man Walking': 35,
 'Across the Sea of Time': 36,
 'It Takes Two': 28953,
 'Clueless': 38,
 'Cry, the Beloved Country': 26505,
 'Richard III': 17651,
 'Dead Presidents': 41,
 'Restoration': 38251,
 'Mortal Kombat': 43,
 'To D

In [37]:
tf = TfidfVectorizer(stop_words='english',min_df=20,binary =True)

In [38]:
tfidf_matrix = tf.fit_transform(data.tags).toarray()

In [39]:
tfidf_matrix.shape

(45047, 7946)

In [40]:
model = NearestNeighbors(n_neighbors=6,algorithm='brute',metric='cosine')

In [41]:
model.fit(tfidf_matrix)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",6
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


In [42]:
def recommend(movie):
    idx = data[data.title == movie].index[0]
    distances ,indices = model.kneighbors(tfidf_matrix[idx].reshape(1,-1))
    indices = indices.reshape(6,)
    distances = distances.reshape(6,)
    movies_list = np.column_stack((indices,distances))
    
    for i in movies_list[1:6]:
        print(data.iloc[int(i[0])].title)



In [43]:
recommend('Toy Story')

Toy Story 2
Toy Story 3
Hawaiian Vacation
Small Fry
Partysaurus Rex


In [44]:
data.to_csv('dataset.csv')

In [45]:
data.head(1)

,id,title,poster_path,tags
0,862,Toy Story,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,anim comedi famili toystorycollect toystorycol...


In [46]:
pickle.dump(data.to_dict(),open('movies_data.pkl','wb'))

In [47]:
pickle.dump(model, open('model.pkl', 'wb'))
pickle.dump(tfidf_matrix, open('tfidf.pkl', 'wb'))

In [51]:
data.poster_path.dtype

<StringDtype(na_value=nan)>